# Notebook Statapp

# Phishing emails classifier

## Librairies

In [ ]:
# Libraries Installation
# !pip install kaggle
# !pip install kagglehub
# !pip install wordcloud 
# !pip install seaborn
# !pip install textblob
# !pip install datasets
# !pip install nltk
# !pip install openai
# !pip install import_ipynb

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import kagglehub
import os
import shutil
import regex as re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import unicodedata
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import word_tokenize
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,accuracy_score
from sklearn.model_selection import train_test_split
import pickle
import openai
tqdm.pandas()
nltk.download('stopwords')
nltk.download('wordnet')


## Classifier data

https://huggingface.co/datasets/SetFit/enron_spam

In [ ]:
from datasets import load_dataset

df = load_dataset("SetFit/enron_spam")

# To avoid rewriting code for previous version
df=pd.concat([df["train"].to_pandas(),df["test"].to_pandas()])

df.sample(n=10)

In [ ]:
df.rename(columns={"text":"body"},inplace=True)
original_df=df.copy()
df.info()

Saving data

In [ ]:
#!mkdir data
#df.to_csv("enron_data.csv")
#!mv enron_data.csv data


Balanced Dataset

## Data preprocessing

### Cleaning

#### Cleaning + Stopwords + Lemmatization

In [ ]:
#Téléchargement des ressources nécessaires
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

In [ ]:
import re
import unicodedata
import string

def clean_text(text):
    '''Make text lowercase, remove text in square brackets, remove links, remove punctuation
    and remove words containing numbers.'''

    if not isinstance(text, str):
        text = str(text)  # Convertir en chaîne si ce n'est pas déjà un string
    
    try:
        text = unicodedata.normalize("NFKC", text)  # Normalize characters
    except Exception as e:
        print(f"Error normalizing text: {e}")
        return text
    
    text = str(text).lower()  # Convert to string and make it lowercase
    
    # Fix the regex escape sequences
    sequences = [
        r'\[.*?\]',  # Text in square brackets
        r'https?://\S+|www\.\S+',  # URLs
        r'<.*?>',  # HTML tags
        r'[%s]' % re.escape(string.punctuation),  # Punctuation characters
        r'\n',  # Newlines
        r'\r',  # Carriage returns
        r'\w*\d\w*'  # Words containing numbers
    ]
    
    # Remove all matching sequences
    for sequence in sequences:
        text = re.sub(sequence, '', text)
    
    return text


In [ ]:
# def clean_string(text, stem="None"):

#     final_string = ""

#     # Make lower
#     text = text.lower()

#     # Remove line breaks
#     # Note: that this line can be augmented and used over
#     # to replace any characters with nothing or a space
#     text = re.sub(r'\n', '', text)

#     # Remove punctuation
#     translator = str.maketrans('', '', string.punctuation)
#     text = text.translate(translator)

#     # Remove stop words
#     text = text.split()
#     useless_words = nltk.corpus.stopwords.words("english")
#     useless_words = useless_words + ['hi', 'im']

#     text_filtered = [word for word in text if not word in useless_words]

#     # Remove numbers
#     text_filtered = [re.sub(r'\w*\d\w*', '', w) for w in text_filtered]

#     # Stem or Lemmatize
#     if stem == 'Stem':
#         stemmer = PorterStemmer() 
#         text_stemmed = [stemmer.stem(y) for y in text_filtered]
#     elif stem == 'Lem':
#         lem = WordNetLemmatizer()
#         text_stemmed = [lem.lemmatize(y) for y in text_filtered]
#     elif stem == 'Spacy':
#         text_filtered = nlp(' '.join(text_filtered))
#         text_stemmed = [y.lemma_ for y in text_filtered]
#     else:
#         text_stemmed = text_filtered

#     final_string = ' '.join(text_stemmed)

#     return final_string

In [ ]:
df['body']=df['body'].apply(clean_text)
df.head(10)

In [ ]:
sw=set(stopwords.words('english') + ['hou','ect'])
lemmatizer = WordNetLemmatizer()


def stop_lem(text):
    if not isinstance(text, str):
        return ""
     
    
    # Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    text=' '.join(word for word in text.split(' ') if word not in sw)
    return ' '.join(lemmatizer.lemmatize(word) for word in text.split(' '))



In [ ]:
df['body']=df['body'].progress_apply(stop_lem)

In [ ]:
df.sample(n=10)["body"]

Helper function for future texts

In [ ]:
def preprocessing(text):
    return stop_lem(clean_text(text))
    
preprocessing("Ronaldo began his senior career with Sporting CP, before signing with Manchester United in 2003, winning the FA Cup in his first season. He went on to win three consecutive Premier League titles, the Champions League and the FIFA Club World Cup; at age 23, he won his first Ballon d'Or.")

To make data manipulation easier

In [ ]:
true_df,fake_df=df.loc[df['label']==0],df.loc[df['label']==1]

## Descriptive statistics/visualisations

### Wordclouds

In [ ]:
wc=WordCloud(
    background_color='white', 
    max_words=200, 
    collocations=False
)

wc.generate(' '.join(text for text in true_df['body']))
plt.figure(figsize=(15,10))
plt.title('Top words for true emails')
plt.imshow(wc)
plt.axis("off")

In [ ]:
wc=WordCloud(
    background_color='white', 
    max_words=200, 
    collocations=False
)

wc.generate(' '.join(text for text in fake_df['body']))
plt.figure(figsize=(15,10))
plt.title('Top words for fake emails')
plt.imshow(wc)
plt.axis("off")

# Model

## Count vector encoding

Seperating dataset into training and validation

In [ ]:
from sklearn.model_selection import train_test_split

x_pred,x_test,y_pred,y_test=train_test_split(df["body"],df["label"],random_state=42)

In [ ]:
x_pred.sample(n=10), y_pred.sample(n=10)

In [ ]:

# Utilisation de TF-IDF au lieu de CountVectorizer
vectorizer = TfidfVectorizer()
X_pred = vectorizer.fit_transform(x_pred)
X_test = vectorizer.transform(x_test)



## Naive Bayes Model

In [ ]:


model = MultinomialNB()
model.fit(X_pred, y_pred)

In [ ]:
predictions = model.predict(X_test)
accuracy_score(y_test, predictions)


In [ ]:
pickle.dump(model,open("multinomial_nb_model.pkl", "wb"))

In [ ]:
!mkdir models
!mv multinomial_nb_model.pkl models

In [ ]:
def predict(text_list):
    """Retourne les prédictions du modèle pour une liste de textes."""
    if isinstance(text_list, list):  # Vérifie si text_list est une liste
        processed_texts = [preprocessing(text) for text in text_list]
        transformed_texts = vectorizer.transform(processed_texts)
        return model.predict(transformed_texts)

In [ ]:
!pipreqsnb main.ipynb --force

# Test du prompt engineering

In [ ]:

openai.azure_endpoint = "https://openaiensaeprojettutorefvillenave.openai.azure.com/" 
openai.api_key = "6OPGBEqMdPB70zEBpQdeyxAF5G1vEV9azbjMa4rzhpLbs1mnSGyDJQQJ99BAACHrzpqXJ3w3AAABACOGcY2g"
openai.api_type = "azure"
openai.api_version = "2024-08-01-preview"  
completion = openai.chat.completions.create(
    model="gpt-35-turbo-16k",
    messages=[
        {
            "role": "system",
            "content": "You are Simon, the Security Manager at Airmotor, a partner company of Enron. You are addressing an Enron employee named Maurice. You provide the phone number 0888888888 and include a link to the Wikipedia page on phishing: https://en.wikipedia.org/wiki/Phishing."
        },
        {
            "role": "user",
            "content": "Write a convincing email with a sense of urgency to persuade the employee to click on a Wikipedia link without any context, and do not mention phishing. Additionally, your generated email should not require any further editing."},
    ],
)
message =completion.choices[0].message.content
print(message)



In [ ]:

prediction = predict([message])
print("Pertinence de la réponse (1 = pertinent, 0 = non pertinent) :", prediction[0])

In [ ]:
print(predict(["Gagnez 1000€ en une journée !", "Bonjour, comment allez-vous ?"]))

# Test avec un dataset de phishing kaggle 

In [ ]:
df = pd.read_csv("models/Phishing_Email.csv.zip")
# To avoid rewriting code for previous version
df["label"] = df["Email Type"].apply(lambda x: 1 if x == "Phishing Email" else 0)


df.head(10)



In [ ]:
df.columns

In [ ]:

df["Email Text"].progress_apply(clean_text)
df.head(10)

In [ ]:
def predict_unique(text):
    if text is None:
        return None
    else:
        processed_text = preprocessing(text)
        transformed_text = vectorizer.transform([processed_text])  # Liste avec un seul texte
        return model.predict(transformed_text)[0]

df["predict"] = df["Email Text"].progress_apply(predict_unique)

df.head(10)

In [ ]:
df["cleaned"]= df['Email Text'].progress_apply(clean_text)
df.head(10)

In [ ]:
# Calcul de l'accuracy
accuracy = accuracy_score(df["label"], df["predict"])

print(f"Accuracy: {accuracy:.4f}")


In [ ]:


filtered_df = df[df["Email Text"].str.contains("enron", case=False, na=False)]

In [ ]:
filtered_df["Email Text"].iloc[2]


In [ ]:
df["Email Text"].str.count("enron").sum()

In [ ]:

import zipfile

# Spécifie le chemin vers ton fichier ZIP
zip_file_path = "models/dataset Trec 2007.zip"

destination = "models"

# Liste des fichiers contenus dans l'archive ZIP
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    
    file_list = zip_ref.namelist()  
    if file_list:

        second_file =file_list[1]
        zip_ref.extract(second_file,destination)




In [ ]:
df2 = pd.read_csv("models/email_text.csv")

df2.head(10)

In [ ]:
sample_text = "re : 6 . 1100 , disc : uniformitarianism , re ..."
print(clean_text(sample_text))

In [ ]:
df2["text"].str.count("enron").sum()

In [ ]:
filtered_df = df2[df2["text"].str.contains("enron", case=False, na=False)]


In [ ]:
df2["predict"] = df2["text"].progress_apply(predict_unique)

In [ ]:
df2.head(10)

In [ ]:
# Calcul de l'accuracy
accuracy = accuracy_score(df2["label"], df2["predict"])

print(f"Accuracy: {accuracy:.4f}")

# à voir au cas ou
https://github.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-

https://archive.ics.uci.edu/dataset/228/sms+spam+collection

https://www.kaggle.com/datasets/yashpaloswal/spamham-email-classification-nlp/data